# yuyv2gray under PYNQ

The `kria_dma_bist_mod` design: AXI DMA MM2S → **`yuyv2gray`** (VHDL) → AXI DMA S2MM.

The block keeps the Y (brightness) bytes of a YUYV stream and drops U and V, so a 640×480 frame goes in as **614 400 bytes** and comes back as a **307 200-byte grayscale image**: one MM2S transfer and one S2MM transfer per frame.

1. synthetic data: exact check of the data, of the length S2MM reports, and that nothing is written past the frame
2. one real webcam frame: FPGA grayscale vs CPU grayscale, shown side by side
3. a short live loop: every frame checked, frame rate measured

In [ ]:
import time
import subprocess
import numpy as np
import cv2
import matplotlib.pyplot as plt
from pynq import Overlay, allocate

W, H      = 640, 480
IN_BYTES  = W * H * 2          # YUYV: 2 bytes per pixel
OUT_BYTES = W * H              # gray: 1 byte per pixel
GUARD     = 64                 # extra rx bytes that must stay untouched
S2MM_LENGTH = 0x58             # DMA register: bytes actually received by S2MM

ol = Overlay('yuyv2gray.bit')
dma = ol.axi_dma_0
print('IP blocks :', list(ol.ip_dict.keys()))
print('max bytes per transfer :', dma.buffer_max_size)

In [ ]:
def fpga_gray(tx, rx):
    """Send a YUYV frame (tx) through yuyv2gray into rx.
    Returns (seconds, bytes received by S2MM). rx is pre-filled with 0xEE so the guard can be checked."""
    rx[:] = 0xEE
    tx.flush()
    rx.flush()
    t0 = time.perf_counter()
    dma.recvchannel.transfer(rx)      # arm S2MM first, with the whole buffer: it stops at tlast
    dma.sendchannel.transfer(tx)
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    elapsed = time.perf_counter() - t0
    rx.invalidate()
    return elapsed, dma.read(S2MM_LENGTH)

tx = allocate(shape=(IN_BYTES,), dtype=np.uint8)
rx = allocate(shape=(OUT_BYTES + GUARD,), dtype=np.uint8)

## 1. Synthetic data

Expected output = every other input byte, starting at byte 0 (`Y0 U Y1 V` → `Y0 Y1`), i.e. `tx[0::2]`.

In [ ]:
fails = 0
for i in range(10):
    tx[:] = np.random.randint(0, 256, IN_BYTES, dtype=np.uint8)
    t, received = fpga_gray(tx, rx)
    ok_data  = np.array_equal(rx[:OUT_BYTES], tx[0::2])
    ok_len   = received == OUT_BYTES
    ok_guard = bool(np.all(rx[OUT_BYTES:] == 0xEE))
    ok = ok_data and ok_len and ok_guard
    fails += not ok
    print(f'run {i}: {"ok  " if ok else "FAIL"}  data {"ok" if ok_data else "BAD"}, '
          f'S2MM received {received} B, guard {"ok" if ok_guard else "OVERWRITTEN"}, '
          f'{t*1e3:5.2f} ms, {IN_BYTES/t/1e6:5.1f} MB/s in')
print('PASS' if fails == 0 else f'FAIL ({fails} runs)')

## 2. One webcam frame

Raw YUYV straight from the camera (no conversion on the CPU). If OpenCV cannot deliver the raw format, `v4l2-ctl` is used instead.
Shut down other notebooks that use the webcam first.

In [ ]:
def open_cam(dev=0):
    """OpenCV capture that returns raw YUYV frames, or None if this OpenCV can't."""
    cap = cv2.VideoCapture(dev, cv2.CAP_V4L2)
    cap.set(cv2.CAP_PROP_FOURCC, cv2.VideoWriter_fourcc(*'YUYV'))
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, W)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, H)
    cap.set(cv2.CAP_PROP_CONVERT_RGB, 0)
    for _ in range(5):                      # let auto-exposure settle
        ok, frame = cap.read()
    if ok and frame is not None and frame.size == IN_BYTES:
        return cap
    cap.release()
    return None

def grab_yuyv_v4l2(dev=0):
    path = '/tmp/frame.yuyv'
    subprocess.run(['v4l2-ctl', '-d', f'/dev/video{dev}',
                    '--set-fmt-video', f'width={W},height={H},pixelformat=YUYV',
                    '--stream-mmap', '--stream-count=5', f'--stream-to={path}'],
                   check=True, capture_output=True)
    return np.fromfile(path, dtype=np.uint8)[-IN_BYTES:]   # last of the 5 frames

cap = open_cam()
if cap is not None:
    ok, frame = cap.read()
    raw, source = np.asarray(frame, dtype=np.uint8).reshape(-1), 'OpenCV'
else:
    raw, source = grab_yuyv_v4l2(), 'v4l2-ctl'
print(f'raw YUYV frame from {source}: {raw.size} bytes')

tx[:] = raw
t, received = fpga_gray(tx, rx)
fpga = np.array(rx[:OUT_BYTES]).reshape(H, W)
cpu  = cv2.cvtColor(raw.reshape(H, W, 2), cv2.COLOR_YUV2GRAY_YUYV)

print(f'S2MM received {received} B in {t*1e3:.2f} ms')
print('FPGA == CPU grayscale:', np.array_equal(fpga, cpu))

In [ ]:
color = cv2.cvtColor(raw.reshape(H, W, 2), cv2.COLOR_YUV2RGB_YUYV)
fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
ax[0].imshow(color);                              ax[0].set_title('webcam (YUYV -> RGB on the CPU)')
ax[1].imshow(fpga, cmap='gray', vmin=0, vmax=255); ax[1].set_title('grayscale from the FPGA')
ax[2].imshow(cv2.absdiff(fpga, cpu), cmap='hot'); ax[2].set_title('|FPGA - CPU| (should be black)')
for a in ax: a.axis('off')
plt.show()

## 3. Live loop

Every frame goes camera → DDR → FPGA → DDR and is checked against the CPU result. The frame rate is set by the camera (30 fps, less in low light); the FPGA part is the `fpga` column.

In [ ]:
N = 60
if cap is None:
    print('OpenCV does not deliver raw YUYV here, live loop skipped')
else:
    t_cap = t_fpga = 0.0
    bad = 0
    t_start = time.perf_counter()
    for i in range(N):
        t0 = time.perf_counter()
        ok, frame = cap.read()
        t_cap += time.perf_counter() - t0
        tx[:] = np.asarray(frame, dtype=np.uint8).reshape(-1)
        t, received = fpga_gray(tx, rx)
        t_fpga += t
        bad += (received != OUT_BYTES) or not np.array_equal(rx[:OUT_BYTES], tx[0::2])
    total = time.perf_counter() - t_start
    print(f'{N} frames, {bad} bad, {N/total:.1f} fps overall')
    print(f'per frame: capture {t_cap/N*1e3:.1f} ms, fpga {t_fpga/N*1e3:.2f} ms')

    plt.figure(figsize=(6, 4.5))
    plt.imshow(np.array(rx[:OUT_BYTES]).reshape(H, W), cmap='gray', vmin=0, vmax=255)
    plt.title('last frame, FPGA grayscale'); plt.axis('off'); plt.show()

In [ ]:
if cap is not None:
    cap.release()
for b in (tx, rx):
    b.freebuffer()